# 🧠 TP3 - Multi-Layer Perceptron (MLP)

In this TP, we will build and train an Artificial Neural Network (ANN) using TensorFlow and Keras.

---

## 📊 Step 1: Data Loading and Exploration

In this section, we:
1. Load the dataset.
2. Explore its structure (rows, columns, data types, etc.).
3. Split it into features (X) and labels (y).
4. Prepare the data for training.


In [24]:
# Step 1: Set random seed for reproducibility
import numpy as np
import tensorflow as tf
import random

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print("✅ Random seeds set")

# Step 2: Load NSL-KDD dataset
import pandas as pd


columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]


df = pd.read_csv('NSL_Binary.csv', names=columns)

print("✅ Dataset loaded successfully")
print("Shape:", df.shape)
df.head()


✅ Random seeds set
✅ Dataset loaded successfully
Shape: (125973, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [25]:
# Step 3: Display info and label distribution
df.info()

print("\nLabel distribution:")
print(df['label'].value_counts())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Data columns (total 43 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   duration                     125973 non-null  int64  
 1   protocol_type                125973 non-null  object 
 2   service                      125973 non-null  object 
 3   flag                         125973 non-null  object 
 4   src_bytes                    125973 non-null  int64  
 5   dst_bytes                    125973 non-null  int64  
 6   land                         125973 non-null  int64  
 7   wrong_fragment               125973 non-null  int64  
 8   urgent                       125973 non-null  int64  
 9   hot                          125973 non-null  int64  
 10  num_failed_logins            125973 non-null  int64  
 11  logged_in                    125973 non-null  int64  
 12  num_compromised              125973 non-null  int64  
 13 

### Q2: How many samples are in the dataset?
There are  25192 samples (rows) in the NSL-KDD dataset.

### Q3: How many features are present (excluding label and difficulty)?
There are 42 features used for training.


### Q5: Why is this distribution important for model training?
If the dataset is imbalanced, the model might learn to predict the majority class more often (e.g., “attack”)
and ignore the minority (“normal”). We must handle this carefully to avoid biased models.


In [26]:
# Part 3:  Data Processing
## Step 1: Remove 'difficulty' column (not useful for training)
df = df.drop('difficulty', axis=1)

print("✅ 'difficulty' column removed")
print("New shape:", df.shape)

## Step 2: Separate features (X) and labels (y)
X = df.drop('label', axis=1)
y = df['label']

print("✅ Features (X) and labels (y) separated")
print("X shape:", X.shape)
print("y shape:", y.shape)


✅ 'difficulty' column removed
New shape: (125973, 42)
✅ Features (X) and labels (y) separated
X shape: (125973, 41)
y shape: (125973,)


In [27]:
# Step 3: Encode categorical features using One-Hot Encoding
X_encoded = pd.get_dummies(X, columns=['protocol_type', 'service', 'flag'])

print("✅ Categorical features encoded successfully")
print("New number of columns after encoding:", X_encoded.shape[1])

# Step 4: Encode labels (Normal -> 0, Attack -> 1)
y_binary = y.apply(lambda x: 0 if x == 'normal' else 1)

print("✅ Labels encoded: Normal → 0, Attack → 1")
print(y_binary.value_counts())


✅ Categorical features encoded successfully
New number of columns after encoding: 122
✅ Labels encoded: Normal → 0, Attack → 1
label
0    67343
1    58630
Name: count, dtype: int64


In [28]:
X_encoded.dtypes

duration          int64
src_bytes         int64
dst_bytes         int64
land              int64
wrong_fragment    int64
                  ...  
flag_S1            bool
flag_S2            bool
flag_S3            bool
flag_SF            bool
flag_SH            bool
Length: 122, dtype: object

In [29]:


# Step 5: Normalize numeric features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

print("✅ Data normalized successfully")


✅ Data normalized successfully


In [30]:
# Step 6: Split data into training and testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_binary, test_size=0.2, random_state=42
)

print("✅ Data split completed")
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)


✅ Data split completed
Training set: (100778, 122)
Testing set: (25195, 122)


### Q6: What encoding technique do you use for categorical variables? Why?
We used **One-Hot Encoding** (via `pd.get_dummies()`).
It transforms categorical values into binary columns so that each category is represented numerically,
which helps the neural network process them correctly.

### Q7: How many features do you have after encoding? Why did the number increase?
After encoding, we have **122 features** (instead of 41).
The number increased because each categorical variable (protocol_type, service, flag)
was split into several binary columns.

### Q8: What type of classification problem is this?
This is a **binary classification** problem — “normal” (0) vs “attack” (1).

### Q9: Why is feature scaling important for neural networks?
Because neural networks use gradient-based optimization.
If features have very different scales, training becomes unstable and convergence is slower.
Scaling helps the model learn faster and better.

### Q10: How many samples are in the training and testing sets?
80% of the samples are used for training and 20% for testing.
→ For NSL-KDD: around 100,778 for training and 25,195 for testing.


# Build 2 ANN models

In [ ]:
# Step 1: Build a function to create models dynamically
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

def build_model(n_hidden_layers, n_neurons, learning_rate, dropout_rate):
    model = Sequential()

    # Input layer
    model.add(Dense(n_neurons, activation='relu', input_shape=(X_train.shape[1],)))

    # Hidden layers
    for _ in range(n_hidden_layers - 1):
        model.add(Dense(n_neurons, activation='relu'))
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))

    # Output layer (binary classification)
    model.add(Dense(1, activation='sigmoid'))

    # Compile model
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

    return model

print("✅ build_model() function ready!")


✅ build_model() function ready!


In [32]:
# Step 2: Define Model 1 (Shallow Network)
model1 = build_model(
    n_hidden_layers=1,
    n_neurons=4,
    learning_rate=0.05,
    dropout_rate=0.0
)

model1.summary()


# Step 3: Define Model 2 (Deep Network)
model2 = build_model(
    n_hidden_layers=3,
    n_neurons=32,
    learning_rate=0.001,
    dropout_rate=0.2
)

model2.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 4)                 492       
                                                                 
 dense_1 (Dense)             (None, 1)                 5         
                                                                 
Total params: 497 (1.94 KB)
Trainable params: 497 (1.94 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_2 (Dense)             (None, 32)                3936      
                                                                 
 dense_3 (Dense)             (None, 32)                1056      
                                                                 
 dr

In [33]:
# Step 1: Train the shallow network
history1 = model1.fit(
    X_train, y_train,
    validation_split=0.2,   # 20% of training data for validation
    epochs=15,
    batch_size=512,
    verbose=1
)

# Evaluate on test data
test_loss1, test_acc1 = model1.evaluate(X_test, y_test, verbose=0)

print(f"\n✅ Shallow network test accuracy: {test_acc1:.4f}")


Epoch 1/15


158/158 [==============================] - 1s 3ms/step - loss: 0.1017 - accuracy: 0.9538 - val_loss: 0.0623 - val_accuracy: 0.9747
Epoch 2/15
158/158 [==============================] - 0s 2ms/step - loss: 0.0553 - accuracy: 0.9787 - val_loss: 0.0572 - val_accuracy: 0.9787
Epoch 3/15
158/158 [==============================] - 0s 2ms/step - loss: 0.0508 - accuracy: 0.9797 - val_loss: 0.0581 - val_accuracy: 0.9788
Epoch 4/15
158/158 [==============================] - 0s 2ms/step - loss: 0.0498 - accuracy: 0.9802 - val_loss: 0.0559 - val_accuracy: 0.9809
Epoch 5/15
158/158 [==============================] - 0s 2ms/step - loss: 0.0479 - accuracy: 0.9821 - val_loss: 0.0525 - val_accuracy: 0.9824
Epoch 6/15
158/158 [==============================] - 0s 2ms/step - loss: 0.0460 - accuracy: 0.9827 - val_loss: 0.0488 - val_accuracy: 0.9829
Epoch 7/15
158/158 [==============================] - 0s 2ms/step - loss: 0.0449 - accuracy: 0.9834 - val_loss: 0.0468 - val_accuracy: 0.9832
Epoc

In [34]:
# Step 2: Train the deep network
history2 = model2.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=64,
    verbose=1
)

# Evaluate on test data
test_loss2, test_acc2 = model2.evaluate(X_test, y_test, verbose=0)

print(f"\n✅ Deep network test accuracy: {test_acc2:.4f}")


Epoch 1/15
1260/1260 [==============================] - 3s 2ms/step - loss: 0.0761 - accuracy: 0.9751 - val_loss: 0.0300 - val_accuracy: 0.9912
Epoch 2/15
1260/1260 [==============================] - 3s 2ms/step - loss: 0.0280 - accuracy: 0.9909 - val_loss: 0.0255 - val_accuracy: 0.9917
Epoch 3/15
1260/1260 [==============================] - 3s 2ms/step - loss: 0.0220 - accuracy: 0.9923 - val_loss: 0.0215 - val_accuracy: 0.9923
Epoch 4/15
1260/1260 [==============================] - 2s 2ms/step - loss: 0.0196 - accuracy: 0.9931 - val_loss: 0.0209 - val_accuracy: 0.9920
Epoch 5/15
1260/1260 [==============================] - 3s 2ms/step - loss: 0.0181 - accuracy: 0.9937 - val_loss: 0.0204 - val_accuracy: 0.9934
Epoch 6/15
1260/1260 [==============================] - 3s 2ms/step - loss: 0.0173 - accuracy: 0.9943 - val_loss: 0.0207 - val_accuracy: 0.9929
Epoch 7/15
1260/1260 [==============================] - 3s 2ms/step - loss: 0.0163 - accuracy: 0.9943 - val_loss: 0.0197 - val_accuracy: